In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [39]:
df = pd.read_csv('cyclone_dataset.csv')

X = df.drop(columns=['Cyclone', 'Pre_existing_Disturbance'])
y = df['Cyclone']

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Feature matrix shape: (2000, 8)
Target distribution:
Cyclone
1    1000
0    1000
Name: count, dtype: int64


In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [41]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n--- 5-Fold Cross Validation Results (Without Data Leakage) ---")
for name, model in models.items():
    X_input = X_train_scaled if name == "Logistic Regression" else X_train
    cv_acc = cross_val_score(model, X_input, y_train, cv=cv, scoring='accuracy')
    cv_roc = cross_val_score(model, X_input, y_train, cv=cv, scoring='roc_auc')
    print(f"{name:22s} | Mean CV Accuracy: {cv_acc.mean()*100:.2f}% | Mean ROC-AUC: {cv_roc.mean():.4f}")


--- 5-Fold Cross Validation Results (Without Data Leakage) ---
Logistic Regression    | Mean CV Accuracy: 99.75% | Mean ROC-AUC: 1.0000
Random Forest          | Mean CV Accuracy: 100.00% | Mean ROC-AUC: 1.0000
Gradient Boosting      | Mean CV Accuracy: 99.81% | Mean ROC-AUC: 1.0000


In [42]:
best_model = models["Random Forest"]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print("\n--- Test Set Evaluation (Random Forest) ---")
print(classification_report(y_test, y_pred, digits=4))
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")


--- Test Set Evaluation (Random Forest) ---
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       200
           1     1.0000    1.0000    1.0000       200

    accuracy                         1.0000       400
   macro avg     1.0000    1.0000    1.0000       400
weighted avg     1.0000    1.0000    1.0000       400

Test Set ROC-AUC Score: 1.0000


In [43]:
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances:")
print(importances.to_string())


Feature Importances:
Ocean_Depth                0.281329
Sea_Surface_Temperature    0.198842
Atmospheric_Pressure       0.192199
Humidity                   0.118134
Latitude                   0.071130
Proximity_to_Coastline     0.068516
Vorticity                  0.041931
Wind_Shear                 0.027919


In [44]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['No Cyclone (0)', 'Cyclone (1)'],
            yticklabels=['No Cyclone (0)', 'Cyclone (1)'])
axes[0].set_title('Test Set Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

Text(28.999999999999986, 0.5, 'True Label')

In [45]:
importances.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Random Forest Feature Importances (Cleaned Data)')
axes[1].set_xlabel('Gini Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

<Figure size 640x480 with 0 Axes>

In [46]:
def predict_cyclone_risk(weather_data: dict):
    input_df = pd.DataFrame([weather_data])
    prob = best_model.predict_proba(input_df)[0][1]
    prediction = int(prob >= 0.5)
    return {
        "Cyclone_Occurrence": bool(prediction),
        "Cyclone_Probability": round(float(prob), 4),
        "Risk_Level": "High" if prob >= 0.7 else ("Moderate" if prob >= 0.4 else "Low")
    }

In [49]:
sample_weather = {
    'Sea_Surface_Temperature': 28.5,
    'Atmospheric_Pressure': 992.0,
    'Humidity': 82.0,
    'Wind_Shear': 11.0,
    'Vorticity': 0.000065,
    'Latitude': 12.5,
    'Ocean_Depth': 220.0,
    'Proximity_to_Coastline': 0.85
}

result = predict_cyclone_risk(sample_weather)
print("\nInference on Sample Observation:")
print(result)


Inference on Sample Observation:
{'Cyclone_Occurrence': True, 'Cyclone_Probability': 1.0, 'Risk_Level': 'High'}


In [50]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("cyclone_dataset.csv")

# Drop target column 'Cyclone'
X = df.drop(columns=["Cyclone", "Pre_existing_Disturbance"])
y = df["Cyclone"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = RandomForestClassifier(
    n_estimators=150, max_depth=8, random_state=42, n_jobs=-1
)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
print(f"5-Fold CV Accuracy: {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std() * 100:.2f}%)")


model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n--- Test Set Evaluation ---")
print(classification_report(y_test, y_pred, digits=4))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")



joblib.dump(model, "cyclone_rf_model.joblib")


joblib.dump(scaler, "cyclone_scaler.joblib")
joblib.dump(list(X.columns), "feature_names.joblib")

print("\nModel, scaler, and feature list saved successfully.")


loaded_model = joblib.load("cyclone_rf_model.joblib")
features = joblib.load("feature_names.joblib")


sample_input = pd.DataFrame(
    [
        {
            "Sea_Surface_Temperature": 28.5,
            "Atmospheric_Pressure": 992.0,
            "Humidity": 85.0,
            "Wind_Shear": 11.0,
            "Vorticity": 0.000065,
            "Latitude": 12.0,
            "Ocean_Depth": 250.0,
            "Proximity_to_Coastline": 0.85,
        }
    ]
)[features]

predicted_class = loaded_model.predict(sample_input)[0]
predicted_prob = loaded_model.predict_proba(sample_input)[0][1]

print(f"\nSample Prediction -> Cyclone: {bool(predicted_class)} (Probability: {predicted_prob:.4f})")

5-Fold CV Accuracy: 100.00% (+/- 0.00%)

--- Test Set Evaluation ---
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       200
           1     1.0000    1.0000    1.0000       200

    accuracy                         1.0000       400
   macro avg     1.0000    1.0000    1.0000       400
weighted avg     1.0000    1.0000    1.0000       400

ROC-AUC Score: 1.0000

Model, scaler, and feature list saved successfully.

Sample Prediction -> Cyclone: True (Probability: 1.0000)
